In [1]:
import pandas as pd
import requests
import json
import sqlite3
from pathlib import Path

In [18]:
DB_PATH = "C:\\Users\\user\\Desktop\\AERO_SENTINEL\\Data\\DB\\aero_sentinel.db"
BASE_URL = "https://api.open-meteo.com/v1/forecast"

London_URL =f"{BASE_URL}?latitude=51.50&longitude=-0.12&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&forecast_days=3"
New_York_URL=f"{BASE_URL}?latitude=40.71&longitude=-74.00&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&forecast_days=3"
Tokyo_URL=f"{BASE_URL}?latitude=35.67&longitude=139.65&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&forecast_days=3"
Sydney_URL=f"{BASE_URL}?latitude=-33.86&longitude=151.20&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&forecast_days=3"
Reykjavik_URL=f"{BASE_URL}?latitude=64.14&longitude=-21.89&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&forecast_days=3"

city_urls = {
    "London": London_URL,
    "New_York": New_York_URL,
    "Tokyo": Tokyo_URL,
    "Sydney": Sydney_URL,
    "Reykjavik": Reykjavik_URL
}

city_data = {}

for city, url in city_urls.items():
    response = requests.get(url)
    city_data[city] = response.json()
print(city_data)


{'London': {'latitude': 51.5, 'longitude': -0.120000124, 'generationtime_ms': 0.11086463928222656, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 1.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h'}, 'hourly': {'time': ['2025-12-13T00:00', '2025-12-13T01:00', '2025-12-13T02:00', '2025-12-13T03:00', '2025-12-13T04:00', '2025-12-13T05:00', '2025-12-13T06:00', '2025-12-13T07:00', '2025-12-13T08:00', '2025-12-13T09:00', '2025-12-13T10:00', '2025-12-13T11:00', '2025-12-13T12:00', '2025-12-13T13:00', '2025-12-13T14:00', '2025-12-13T15:00', '2025-12-13T16:00', '2025-12-13T17:00', '2025-12-13T18:00', '2025-12-13T19:00', '2025-12-13T20:00', '2025-12-13T21:00', '2025-12-13T22:00', '2025-12-13T23:00', '2025-12-14T00:00', '2025-12-14T01:00', '2025-12-14T02:00', '2025-12-14T03:00', '2025-12-14T04:00', '2025-12-14T05:00', '2025-12-14T06:00', '2025-12-14T07:00', '2025-12-14T08:00', '2025-12-14

In [1]:
def create_sql_dat(DB_PATH):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute(
        """DROP TABLE IF EXISTS forecasts"""
    )

    cur.execute("""
        CREATE TABLE IF NOT EXISTS forecasts(
                id INTEGER PRIMARY KEY AUTOINCROMENT
                city TEXT,
                forecast_time TIME,
                temp_c TEXT,
                humidity INTEGER,
                wind_kph REAL
            )
                
                
                
                """)
    conn.commit()
    return conn, cur
conn, cur = create_sql_dat(DB_PATH)

NameError: name 'DB_PATH' is not defined

In [90]:
def insert_data(cur, city_data):
    for city, value in city_data.items():
       for i, time in enumerate(value['hourly']['time']):
        forecast_time = value['hourly']['time'][i]
        temp_c=value['hourly']['temperature_2m'][i]
        humidity=value['hourly']['relative_humidity_2m'][i]
        wind_kph=value['hourly']['wind_speed_10m'][i]

        cur.execute("""
                INSERT INTO forecasts (city, forecast_time, temp_c,  humidity, wind_kph)
                VALUES (?, ?, ?, ?, ?)
            """, (city, forecast_time, temp_c,  humidity, wind_kph))


insert_data(cur, city_data)
conn.commit()

In [130]:
df=pd.read_sql("SELECT * FROM forecasts", conn)
df

,city,forecast_time,temp_c,humidity,wind_kph
0,London,2025-12-13T00:00,7.5,91,7.1
1,London,2025-12-13T01:00,7.1,91,5.8
2,London,2025-12-13T02:00,6.7,91,5.7
3,London,2025-12-13T03:00,6.4,92,6.0
4,London,2025-12-13T04:00,6.1,94,6.0
...,...,...,...,...,...
355,Reykjavik,2025-12-15T19:00,0.3,63,8.6
356,Reykjavik,2025-12-15T20:00,-0.9,65,10.4
357,Reykjavik,2025-12-15T21:00,-1.6,66,9.7
358,Reykjavik,2025-12-15T22:00,-1.2,62,4.0


In [169]:
city_max =df.groupby('city')["temp_c"].agg("max")
city_max.idxmax()

'Sydney'

In [128]:
wind_max_kph = df.loc[df["wind_kph"].idxmax()]
print(wind_max_kph['city'])
print(wind_max_kph["wind_kph"])

Sydney
28.3


In [145]:
df['temp_c']=pd.to_numeric(df['temp_c'])
city_temp =df.groupby('city')['temp_c'].mean()
city_temp


city
London        8.820833
New_York     -2.629167
Reykjavik     1.770833
Sydney       22.416667
Tokyo         6.094444
Name: temp_c, dtype: float64

In [162]:
df['temp_c']=pd.to_numeric(df['temp_c'])
city_status=df.groupby('city')['temp_c'].agg(
    diff= lambda d: d.max()-d.min()
    )
city_status.idxmax()

diff    Sydney
dtype: object

In [166]:
humidity =df.loc[df["humidity"]>90, 'humidity'].count()
humidity

np.int64(45)

In [ ]:
import pandas as pd
import sqlite3


# Load CSV files
products = pd.read_csv(rC:\\Users\\user\\Desktop\\AERO_SENTINEL\\Data\\inventory_noisy.csv")
inventory = pd.read_csv("C:\\Users\user\\Desktop\\AERO_SENTINEL\Data\\products_noisy.csv")

# Fill missing numeric values with mean
products.fillna(products.mean(numeric_only=True), inplace=True)
inventory.fillna(inventory.mean(numeric_only=True), inplace=True)

# Validation: ProductID in inventory must exist in products
inventory = inventory[inventory["ProductID"].isin(products["ProductID"])]



conn = sqlite3.connect("RetailSystem.db")
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS Sales;
DROP TABLE IF EXISTS Inventory;
DROP TABLE IF EXISTS Products;

CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY,
    ProductName TEXT,
    Category TEXT,
    Price REAL
);

CREATE TABLE Inventory (
    InventoryID INTEGER PRIMARY KEY,
    ProductID INTEGER,
    WarehouseCode TEXT,
    StockLevel INTEGER,
    FOREIGN KEY (ProductID) REFERENCES Products(ProductID)
);

CREATE TABLE Sales (
    SaleID INTEGER PRIMARY KEY AUTOINCREMENT,
    ProductID INTEGER,
    QuantitySold INTEGER,
    SaleDate TEXT DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (ProductID) REFERENCES Products(ProductID)
);
""")


# PART 3: INSERT DATA INTO DATABASE


products.to_sql("Products", conn, if_exists="append", index=False)
inventory.to_sql("Inventory", conn, if_exists="append", index=False)

conn.commit()


# PART 4A: TRIGGER - AUTOMATIC STOCK UPDATE


cursor.execute("""
CREATE TRIGGER IF NOT EXISTS UpdateStockAfterSale
AFTER INSERT ON Sales
BEGIN
    UPDATE Inventory
    SET StockLevel = StockLevel - NEW.QuantitySold
    WHERE ProductID = NEW.ProductID;
END;
""")

conn.commit()


# PART 4B: VIEW - CATEGORY REVENUE SUMMARY


cursor.execute("""
CREATE VIEW IF NOT EXISTS CategoryRevenueSummary AS
SELECT
    p.Category,
    SUM(p.Price * i.StockLevel) AS TotalPotentialRevenue
FROM Products p
JOIN Inventory i ON p.ProductID = i.ProductID
GROUP BY p.Category;
""")

conn.commit()


# PART 4C: COMPLEX UPDATE - PRICE DISCOUNT

cursor.execute("""
UPDATE Products
SET Price = Price * 0.8
WHERE ProductID IN (
    SELECT ProductID
    FROM Inventory
    WHERE WarehouseCode = 'WH-A'
      AND StockLevel < 40
);
""")

conn.commit()


# OPTIONAL TEST: INSERT SALE TO TRIGGER STOCK UPDATE


cursor.execute("""
INSERT INTO Sales (ProductID, QuantitySold)
VALUES (1, 5);
""")

conn.commit()


# CLOSE CONNECTION


conn.close()

print("✅ RetailSystem.db successfully created and populated.")
